# Fine-tuning learning-rate sweep, on the already-validated TAPT checkpoint

`FT_LR = 1e-5` (`docs/xlmrlarge_tapt_model.md` / `docs/xlmrlarge_model.md`) was chosen a priori as a "known-safe" value for large-transformer fine-tuning, smaller than the base models' 2e-5, out of caution about large-model instability -- **never swept.** Since then, every run in this project on bf16/L4 has been fully stable (zero NaN losses), so the caution behind picking a smaller-than-default LR may not have been necessary.

**This sweep tests the fine-tuning LR only, not the MLM phase** -- it reuses the already-saved `mlm_epoch_10` checkpoint from the epoch sweep directly (Phase 1 is skipped entirely, saving ~53 min), since that sweep already confirmed 10 MLM epochs is the right setting (no candidate in `{2, 5, 10}` beat it). Only Phase 2 (5-fold classification CV, ~1h per LR candidate) runs here.

**Grid: `FT_LR_GRID = [5e-6, 1e-5, 2e-5]`**:
- `5e-6` is the documented "F1 collapse" fallback value -- worth checking if it's actually *better* on its own merits, not just a safety net for failure cases.
- `1e-5` is the current deployed anchor.
- `2e-5` is the base-size models' LR -- worth checking whether the large-model caution was actually necessary now that bf16 has proven fully stable throughout this project.

**Pre-registered adoption gate** (same discipline as the epoch sweep): a candidate only clears the gate if its mean 5-fold CV beats **this sweep's own `FT_LR=1e-5` point** (paired comparison, same folds/seed/hardware, cancels run-to-run GPU noise) by more than the ~0.006 documented noise floor.

**Run the smoke-test cell first** (below) -- same reasoning as the epoch-sweep notebook: this pipeline hasn't executed in this exact configuration (fixed MLM checkpoint, varying LR) even though the underlying `run_fold` logic was already validated there.

**Compliance**: `test.csv` is never loaded. Only `train.csv` and the already-saved `mlm_epoch_10` checkpoint are used.

## Setup

Select **L4 GPU** runtime before running.

In [1]:
!pip install -q transformers==5.12.1 accelerate datasets


In [2]:
import sys
import torch
import transformers

print(f"Python: {sys.version}")
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_available(), "No GPU -- select a GPU runtime before continuing"
assert torch.cuda.is_bf16_supported(), "GPU doesn't support bf16 -- switch to an L4 runtime"


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.11.0+cu128
transformers: 5.12.1
CUDA available: True
GPU: NVIDIA L4
bf16 supported: True


## Load data and the existing `mlm_epoch_10` checkpoint

Same folder convention as the epoch-sweep notebook -- `MLM_CHECKPOINT_DIR` should already exist on Drive from that run.

In [3]:
import os

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/apar_data"
EPOCH_SWEEP_DIR = "/content/drive/MyDrive/apar_xlmrlarge_tapt_epoch_sweep"  # from the epoch sweep
MLM_CHECKPOINT_DIR = f"{EPOCH_SWEEP_DIR}/mlm_epoch_10"  # reused directly, Phase 1 not re-run
OUTPUT_DIR = "/content/drive/MyDrive/apar_xlmrlarge_tapt_lr_sweep"  # create this folder yourself before running

assert os.path.isdir(MLM_CHECKPOINT_DIR), (
    f"{MLM_CHECKPOINT_DIR} not found -- confirm the epoch sweep's mlm_epoch_10 checkpoint "
    "is still on Drive before running this notebook."
)


Mounted at /content/drive


In [4]:
import re

import pandas as pd
from sklearn.model_selection import StratifiedKFold

RANDOM_SEED = 42
MAX_LENGTH = 96
TAG_MISSING_VALUE = "missing"
LABELS = ["technical_support", "customer_support", "other"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
N_SPLITS = 5


def clean_text(text: str) -> str:
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
train_df["feedback"] = train_df["feedback"].astype(str).apply(clean_text)
train_df["tag"] = train_df["tag"].fillna(TAG_MISSING_VALUE)
train_df["input_text"] = train_df["tag"] + " | " + train_df["feedback"]
train_df["label_id"] = train_df["label"].map(LABEL2ID)

print(f"Loaded {len(train_df)} rows (train.csv only -- test.csv is never loaded in this notebook)")
print(train_df["label"].value_counts())

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
FOLDS = list(skf.split(train_df, train_df["label_id"]))


Loaded 17769 rows (train.csv only -- test.csv is never loaded in this notebook)
label
technical_support    10642
other                 3800
customer_support      3327
Name: count, dtype: int64


## Smoke test -- run this before the real sweep

Loads the real `mlm_epoch_10` checkpoint (confirms it's valid and reachable) and fine-tunes it on 100 toy rows for 1 epoch, using the exact same collator pattern the real sweep uses. Throwaway local scratch dir, never touches `OUTPUT_DIR`.

In [ ]:
import shutil

import numpy as np
import torch
from datasets import Dataset as _SmokeDataset
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

_SMOKE_DIR = "/content/smoke_test_scratch"
shutil.rmtree(_SMOKE_DIR, ignore_errors=True)
os.makedirs(_SMOKE_DIR, exist_ok=True)

print(f"Smoke test starting -- loading real checkpoint from {MLM_CHECKPOINT_DIR} ...")

_smoke_df = train_df.sample(n=100, random_state=RANDOM_SEED).reset_index(drop=True)
_split = int(0.8 * len(_smoke_df))
_train_idx, _val_idx = np.arange(_split), np.arange(_split, len(_smoke_df))

_tok = AutoTokenizer.from_pretrained(MLM_CHECKPOINT_DIR)
_model = AutoModelForSequenceClassification.from_pretrained(MLM_CHECKPOINT_DIR, num_labels=len(LABELS))
print("Checkpoint loaded OK.")

_train_ds = _SmokeDataset.from_dict({
    "text": _smoke_df["input_text"].iloc[_train_idx].tolist(),
    "labels": _smoke_df["label_id"].iloc[_train_idx].tolist(),
}).map(lambda b: _tok(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True, remove_columns=["text"])
_val_ds = _SmokeDataset.from_dict({
    "text": _smoke_df["input_text"].iloc[_val_idx].tolist(),
    "labels": _smoke_df["label_id"].iloc[_val_idx].tolist(),
}).map(lambda b: _tok(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True, remove_columns=["text"])

_trainer = Trainer(
    model=_model,
    args=TrainingArguments(
        output_dir=f"{_SMOKE_DIR}/ft_scratch", num_train_epochs=1,
        per_device_train_batch_size=8, learning_rate=1e-5, bf16=True,
        save_strategy="no", report_to="none", logging_steps=100, seed=RANDOM_SEED,
    ),
    train_dataset=_train_ds,
    eval_dataset=_val_ds,
    data_collator=lambda batch: _tok.pad(batch, return_tensors="pt"),
)
_trainer.train()
_val_logits = _trainer.predict(_val_ds).predictions
_val_preds = np.argmax(_val_logits, axis=1)
_val_true = _smoke_df["label_id"].iloc[_val_idx].to_numpy()
_smoke_f1 = f1_score(_val_true, _val_preds, average="macro")
print(f"Fine-tune smoke OK: F1={_smoke_f1:.3f} (meaningless on 20 rows, just confirms no crash).")

del _model, _trainer
torch.cuda.empty_cache()
shutil.rmtree(_SMOKE_DIR, ignore_errors=True)

print("\nSMOKE TEST PASSED -- safe to run the real sweep cells below.")


Smoke test starting -- loading real checkpoint from /content/drive/MyDrive/apar_xlmrlarge_tapt_epoch_sweep/mlm_epoch_10 ...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/apar_xlmrlarge_tapt_epoch_sweep/mlm_epoch_10
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint loaded OK.


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Step,Training Loss


Fine-tune smoke OK: F1=0.222 (meaningless on 20 rows, just confirms no crash).

SMOKE TEST PASSED -- safe to run the real sweep cells below.


## The sweep: 5-fold CV per learning rate, all from the same fixed `mlm_epoch_10` checkpoint

In [5]:
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset as HFDataset
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

FT_LR_GRID = [5e-6, 1e-5, 2e-5, 3e-5]  # 1e-5 is the current deployed anchor
FT_EPOCHS = 5  # unchanged from the established recipe -- only LR varies here
FT_BATCH = 16
FT_GRAD_ACCUM = 2

class_weights = compute_class_weight(
    "balanced", classes=np.arange(len(LABELS)), y=train_df["label_id"].to_numpy()
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_t.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def run_fold(fold_idx, train_idx, val_idx, lr):
    tokenizer = AutoTokenizer.from_pretrained(MLM_CHECKPOINT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MLM_CHECKPOINT_DIR, num_labels=len(LABELS))

    train_ds = HFDataset.from_dict({
        "text": train_df["input_text"].iloc[train_idx].tolist(),
        "labels": train_df["label_id"].iloc[train_idx].tolist(),
    }).map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True, remove_columns=["text"])
    val_ds = HFDataset.from_dict({
        "text": train_df["input_text"].iloc[val_idx].tolist(),
        "labels": train_df["label_id"].iloc[val_idx].tolist(),
    }).map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True, remove_columns=["text"])

    args = TrainingArguments(
        output_dir=f"/content/ft_scratch/fold{fold_idx}",
        num_train_epochs=FT_EPOCHS,
        per_device_train_batch_size=FT_BATCH,
        gradient_accumulation_steps=FT_GRAD_ACCUM,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        bf16=True,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        seed=RANDOM_SEED,
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=lambda batch: tokenizer.pad(batch, return_tensors="pt"),
    )
    trainer.train()

    val_logits = trainer.predict(val_ds).predictions
    val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
    val_preds = val_probs.argmax(axis=1)
    val_true = train_df["label_id"].iloc[val_idx].to_numpy()
    fold_f1 = f1_score(val_true, val_preds, average="macro")

    del model, trainer
    torch.cuda.empty_cache()
    return fold_f1, val_probs


import json

RESULTS_PATH = f"{OUTPUT_DIR}/lr_sweep_results.json"

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        sweep_results = json.load(f)
    print(f"Resumed {len(sweep_results)} completed LR(s): {list(sweep_results.keys())}")
else:
    sweep_results = {}

for lr in FT_LR_GRID:
    key = str(lr)
    if key in sweep_results:
        print(f"Skipping lr={lr} -- already completed.")
        continue

    print(f"\n{'='*70}\nRunning 5-fold CV for lr={lr}\n{'='*70}")
    fold_f1s = []
    oof_probs = np.zeros((len(train_df), len(LABELS)))
    for fold_idx, (train_idx, val_idx) in enumerate(FOLDS, start=1):
        f1, probs = run_fold(fold_idx, train_idx, val_idx, lr)
        fold_f1s.append(f1)
        oof_probs[val_idx] = probs
        print(f"  lr={lr}  fold={fold_idx}  Macro F1={f1:.4f}")

    mean_f1 = float(np.mean(fold_f1s))
    std_f1 = float(np.std(fold_f1s))
    print(f"lr={lr}: mean={mean_f1:.4f} (+/- {std_f1:.4f})")

    sweep_results[key] = {"fold_f1": fold_f1s, "mean": mean_f1, "std": std_f1}
    with open(RESULTS_PATH, "w") as f:
        json.dump(sweep_results, f, indent=2)
    np.save(f"{OUTPUT_DIR}/oof_probs_lr_{lr}.npy", oof_probs)
    print(f"Saved results + OOF probs for lr={lr} to Drive.")


Resumed 3 completed LR(s): ['5e-06', '1e-05', '2e-05']
Skipping lr=5e-06 -- already completed.
Skipping lr=1e-05 -- already completed.
Skipping lr=2e-05 -- already completed.

Running 5-fold CV for lr=3e-05


Loading weights:   0%|          | 0/389 [00:01<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/apar_xlmrlarge_tapt_epoch_sweep/mlm_epoch_10
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/14215 [00:00<?, ? examples/s]

Map:   0%|          | 0/3554 [00:00<?, ? examples/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/apar_xlmrlarge_tapt_epoch_sweep/mlm_epoch_10
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/14216 [00:00<?, ? examples/s]

Map:   0%|          | 0/3553 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.631944,0.292872
2,0.521193,0.322546
3,0.330511,0.353128
4,0.199233,0.440666
5,0.120610,0.532456


  lr=3e-05  fold=5  Macro F1=0.9167
lr=3e-05: mean=0.9245 (+/- 0.0047)
Saved results + OOF probs for lr=3e-05 to Drive.


## Results and adoption gate

Same paired-comparison logic as the epoch sweep: compares candidates against **this sweep's own `lr=1e-5` point** (same folds/seed/hardware -- noise cancels), not a cross-session number.

In [6]:
NOISE_FLOOR = 0.006

print(f"{'lr':>10s}  {'mean F1':>8s}  {'std':>7s}  {'per-fold'}")
for lr_str, r in sorted(sweep_results.items(), key=lambda kv: float(kv[0])):
    flag = " <- anchor (current deployed LR)" if float(lr_str) == 1e-5 else ""
    print(f"{lr_str:>10s}  {r['mean']:8.4f}  {r['std']:7.4f}  "
          f"{[round(x,4) for x in r['fold_f1']]}{flag}")
print()

assert "1e-05" in sweep_results, "lr=1e-5 must be included to serve as the in-run anchor"
anchor_f1 = sweep_results["1e-05"]["mean"]

best_lr, best_mean = None, -1
for lr_str, r in sweep_results.items():
    if r["mean"] > best_mean:
        best_mean, best_lr = r["mean"], lr_str

print(f"In-run anchor (lr=1e-5): {anchor_f1:.4f}")
print(f"Best candidate: lr={best_lr}, mean={best_mean:.4f}, delta vs. anchor {best_mean - anchor_f1:+.4f}")

gate_passed = (best_lr != "1e-05") and (best_mean - anchor_f1) > NOISE_FLOOR
if gate_passed:
    print(f"\nGATE PASSED: lr={best_lr} exceeds the anchor by {best_mean - anchor_f1:+.4f}, "
          f"above the {NOISE_FLOOR} noise floor.")
    print("-> Worth proceeding to the final retrain cell below.")
else:
    print(f"\nGATE NOT PASSED: no candidate clearly exceeds the lr=1e-5 anchor by more than "
          f"{NOISE_FLOOR}.")
    print("-> Treat as noise / a flat result. Keep the existing lr=1e-5 model.")


        lr   mean F1      std  per-fold
     5e-06    0.9232   0.0052  [0.931, 0.9202, 0.9268, 0.9217, 0.9161]
     1e-05    0.9260   0.0054  [0.9329, 0.92, 0.9289, 0.9289, 0.9192] <- anchor (current deployed LR)
     2e-05    0.9284   0.0044  [0.9332, 0.9247, 0.9332, 0.9287, 0.9222]
     3e-05    0.9245   0.0047  [0.9294, 0.924, 0.9294, 0.9231, 0.9167]

In-run anchor (lr=1e-5): 0.9260
Best candidate: lr=2e-05, mean=0.9284, delta vs. anchor +0.0024

GATE NOT PASSED: no candidate clearly exceeds the lr=1e-5 anchor by more than 0.006.
-> Treat as noise / a flat result. Keep the existing lr=1e-5 model.


## Final retrain (only if the gate passed)

Self-contained -- pastable into a fresh session. Set `WINNING_LR` from the gate cell's output.

In [6]:
# --- Self-contained: paste into a fresh Colab session (L4 GPU runtime) if needed ---
import os
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset as HFDataset
from google.colab import drive
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/apar_data"
EPOCH_SWEEP_DIR = "/content/drive/MyDrive/apar_xlmrlarge_tapt_epoch_sweep"
MLM_CHECKPOINT_DIR = f"{EPOCH_SWEEP_DIR}/mlm_epoch_10"
OUTPUT_DIR = "/content/drive/MyDrive/apar_xlmrlarge_tapt_lr_sweep"

WINNING_LR = 2e-5  # <-- SET THIS from the gate cell's output before running

RANDOM_SEED = 42
MAX_LENGTH = 96
TAG_MISSING_VALUE = "missing"
LABELS = ["technical_support", "customer_support", "other"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
FT_EPOCHS, FT_BATCH, FT_GRAD_ACCUM = 5, 16, 2


def clean_text(text: str) -> str:
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
train_df["feedback"] = train_df["feedback"].astype(str).apply(clean_text)
train_df["tag"] = train_df["tag"].fillna(TAG_MISSING_VALUE)
train_df["input_text"] = train_df["tag"] + " | " + train_df["feedback"]
train_df["label_id"] = train_df["label"].map(LABEL2ID)

class_weights = compute_class_weight(
    "balanced", classes=np.arange(len(LABELS)), y=train_df["label_id"].to_numpy()
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(weight=class_weights_t.to(logits.device))(logits, labels)
        return (loss, outputs) if return_outputs else loss


tokenizer = AutoTokenizer.from_pretrained(MLM_CHECKPOINT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(
    MLM_CHECKPOINT_DIR, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

full_ds = HFDataset.from_dict({
    "text": train_df["input_text"].tolist(),
    "labels": train_df["label_id"].tolist(),
}).map(
    lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["text"],
)

args = TrainingArguments(
    output_dir="/content/final_scratch",
    num_train_epochs=FT_EPOCHS,
    per_device_train_batch_size=FT_BATCH,
    gradient_accumulation_steps=FT_GRAD_ACCUM,
    learning_rate=WINNING_LR,
    bf16=True,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    seed=RANDOM_SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=full_ds,
    data_collator=lambda batch: tokenizer.pad(batch, return_tensors="pt"),
)
trainer.train()

final_dir = f"{OUTPUT_DIR}/final_model_lr_{WINNING_LR}"
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
n_params = sum(p.numel() for p in model.parameters())
print(f"Saved final model to {final_dir}")
print(f"Param count: {n_params:,} (reference: 559,893,507 -- must match)")
print(f"Saved id2label: {model.config.id2label}  (should show real label names, not LABEL_0/1/2)")

oof_probs = np.load(f"{OUTPUT_DIR}/oof_probs_lr_{WINNING_LR}.npy")
oof_df = train_df[["id", "label"]].copy()
for i, label in enumerate(LABELS):
    oof_df[f"prob_{label}"] = oof_probs[:, i]
oof_df.to_csv(f"{OUTPUT_DIR}/xlmrlarge_tapt_lr{WINNING_LR}_oof_predictions.csv", index=False)
print("Saved OOF predictions CSV.")
print()
print("Next steps: download final_model/ to models/xlmrlarge_tapt_lr<LR>_model/ and the OOF CSV")
print("to artifacts/. Re-run notebooks/blend_tapt_tuning.ipynb (or the nested weight retune in")
print("notebooks/lr_sweep_blend_check.ipynb) before deploying -- a different solo model may")
print("shift the optimal blend weight away from W_TAPT=0.4.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/apar_xlmrlarge_tapt_epoch_sweep/mlm_epoch_10
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/17769 [00:00<?, ? examples/s]

Step,Training Loss
50,2.013372
100,0.991260
150,0.685980
200,0.686019
250,0.696506
300,0.689768
350,0.666884
400,0.564040
450,0.632496
500,0.558055


Step,Training Loss
50,2.013372
100,0.991260
150,0.685980
200,0.686019
250,0.696506
300,0.689768
350,0.666884
400,0.564040
450,0.632496
500,0.558055


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model to /content/drive/MyDrive/apar_xlmrlarge_tapt_lr_sweep/final_model_lr_2e-05
Param count: 559,893,507 (reference: 559,893,507 -- must match)
Saved OOF predictions CSV.

Next steps: download final_model/ to models/xlmrlarge_tapt_model/ and the OOF CSV to
artifacts/. Re-run notebooks/blend_tapt_tuning.ipynb before deploying -- a different
solo model may shift the optimal blend weight away from W_TAPT=0.4.
